
---

## 一、核心背景：深度学习的两大痛点
到目前为止，我们用的都是**可完全放入内存（RAM）**的小数据集。但深度学习系统常需要在**超大规模数据集**上训练，这类数据无法一次性载入内存，会带来两个核心难题：
1.  **高效读取**：如何流式读取、分批加载超大数据集，避免内存溢出
2.  **灵活预处理**：如何对数据做归一化、编码（文本/分类特征）等操作，同时不影响训练效率

TensorFlow 的 **Data API** 就是为解决这两个问题而生的，而且能和 `tf.keras` 无缝协同工作。

---

## 二、Data API 核心能力
### 1. 核心优势
- 只需创建一个**数据集对象**，定义「数据来源」和「转换规则」，TensorFlow 会自动处理多线程、队列、批处理、预取等所有底层细节，无需手动实现
- 完美适配 `tf.keras`，可直接用于模型训练

### 2. 支持的数据源
Data API 支持几乎所有主流数据格式/来源：
| 数据源类型 | 说明 |
|------------|------|
| 文本文件 | 如 CSV 文件 |
| 二进制文件 | 固定大小记录的二进制文件 |
| TFRecord 格式 | TensorFlow 原生的高效二进制格式，支持任意大小记录，开源协议缓冲区 |
| 数据库 | 支持从 SQL 数据库读取 |
| 开源扩展 | 如 Google BigQuery 等云服务数据源 |

---

## 三、数据预处理方案
大规模数据不仅要读取，还需要做预处理，常见需求和解决方案如下：
### 1. 常见预处理需求
- 数值特征：归一化/标准化
- 文本特征：词袋编码、TF-IDF
- 分类特征：独热编码、嵌入（Embedding，可训练的密集向量，用于表示类别/令牌）

### 2. 两种实现方式
| 方案 | 说明 |
|------|------|
| 自定义预处理层 | 编写自定义 Keras 层，灵活实现复杂预处理逻辑 |
| Keras 标准预处理层 | 使用 Keras 内置的标准化层（如 `Normalization`、`CategoryEncoding` 等），开箱即用，无需手动实现 |

---

## 四、TensorFlow 生态相关工具
### 1. TF 转换（tf.Transform）
- **核心作用**：编写单个预处理函数，可在完整训练集上以**批处理模式**运行（加速预处理），之后导出为 TF 函数，集成到训练模型中
- **部署优势**：模型部署到生产环境后，可对新实例做**即时预处理**，保证训练和推理的预处理逻辑完全一致，避免数据漂移

### 2. TF 数据集（TFDS，TensorFlow Datasets）
- **核心作用**：提供便捷的 API，一键下载各类常见数据集（包括 ImageNet 这类超大规模数据集）
- **兼容性**：下载后的数据集可直接用 Data API 进行操作、批处理、预处理，无缝对接训练流程

---

## 五、本章学习内容预告
本章将详细讲解：
1.  **Data API**：如何用它加载、处理大规模数据集
2.  **TFRecord 格式**：高效存储和读取数据的方法
3.  **自定义预处理层**：如何编写灵活的预处理逻辑
4.  **Keras 标准预处理层**：如何开箱即用完成数据预处理

---

## 六、核心知识点总结（记忆版）
| 模块 | 核心要点 |
|------|----------|
| Data API | 解决超大数据集的流式读取+预处理，自动处理多线程/批处理/预取，无缝对接 tf.keras |
| 数据源 | 支持 CSV、二进制、TFRecord、SQL、BigQuery 等几乎所有主流来源 |
| 预处理 | 数值归一化、文本/分类编码，可通过自定义层或 Keras 标准层实现 |
| tf.Transform | 统一训练/推理的预处理逻辑，避免数据漂移，加速批处理 |
| TFDS | 一键下载常见数据集，直接用 Data API 操作 |

---

## 七、一句话总结
这部分是本章的开篇，核心是介绍 TensorFlow 针对**超大规模深度学习**的解决方案：用 Data API 解决大数据读取难题，用预处理层/TF 转换解决数据预处理问题，配合 TFDS 快速获取数据集，构建完整的大数据训练 pipeline。

---

# 13.1数据API

In [55]:

import tensorflow as tf
# 在RAM中完全创建一个数据集
X = tf.range(10)
dataset = tf.data.Dataset.from_tensor_slices(X)
dataset


<_TensorSliceDataset element_spec=TensorSpec(shape=(), dtype=tf.int32, name=None)>

In [56]:
#遍历数据集的元素
for item in dataset:
    print(item)

tf.Tensor(0, shape=(), dtype=int32)
tf.Tensor(1, shape=(), dtype=int32)
tf.Tensor(2, shape=(), dtype=int32)
tf.Tensor(3, shape=(), dtype=int32)
tf.Tensor(4, shape=(), dtype=int32)
tf.Tensor(5, shape=(), dtype=int32)
tf.Tensor(6, shape=(), dtype=int32)
tf.Tensor(7, shape=(), dtype=int32)
tf.Tensor(8, shape=(), dtype=int32)
tf.Tensor(9, shape=(), dtype=int32)


## 13.1.1 链式转换

In [57]:
#链式转换
dataset = dataset.repeat(3).batch(7)
for item in dataset:
    print(item)

tf.Tensor([0 1 2 3 4 5 6], shape=(7,), dtype=int32)
tf.Tensor([7 8 9 0 1 2 3], shape=(7,), dtype=int32)
tf.Tensor([4 5 6 7 8 9 0], shape=(7,), dtype=int32)
tf.Tensor([1 2 3 4 5 6 7], shape=(7,), dtype=int32)
tf.Tensor([8 9], shape=(2,), dtype=int32)


In [58]:
# 删除最终批次
import tensorflow as tf

#  重新创建原始数据集
X = tf.range(10)
dataset = tf.data.Dataset.from_tensor_slices(X)

dataset = dataset.repeat(3).batch(7, drop_remainder=True)

#  遍历输出
for item in dataset:
    print(item)

tf.Tensor([0 1 2 3 4 5 6], shape=(7,), dtype=int32)
tf.Tensor([7 8 9 0 1 2 3], shape=(7,), dtype=int32)
tf.Tensor([4 5 6 7 8 9 0], shape=(7,), dtype=int32)
tf.Tensor([1 2 3 4 5 6 7], shape=(7,), dtype=int32)


In [59]:
# 变换元素
dataset = dataset.map(lambda x:x*2 )

In [60]:
# 将unbatch()函数应用于数据集
dataset = dataset.apply(tf.data.experimental.unbatch())

In [61]:
# 简单过滤数据集
dataset = dataset.filter(lambda x:x<10)

In [62]:
# 查看数据集中的元素
for item in dataset.take(3):
    print(item)

tf.Tensor(0, shape=(), dtype=int32)
tf.Tensor(2, shape=(), dtype=int32)
tf.Tensor(4, shape=(), dtype=int32)


## 13.1.2 乱序数据

In [63]:
# 使用shuffle方法对实例进行乱序
dataset = tf.data.Dataset.range(10).repeat(3)
dataset = dataset.shuffle(buffer_size=5,seed=42).batch(7)
for item in dataset:
    print(item)

tf.Tensor([0 2 3 6 7 9 4], shape=(7,), dtype=int64)
tf.Tensor([5 0 1 1 8 6 5], shape=(7,), dtype=int64)
tf.Tensor([4 8 7 1 2 3 0], shape=(7,), dtype=int64)
tf.Tensor([5 4 2 7 8 9 9], shape=(7,), dtype=int64)
tf.Tensor([3 6], shape=(2,), dtype=int64)


In [64]:
#在每次迭代中重用相同的顺序
dataset = tf.data.Dataset.range(10).repeat(3)
dataset = dataset.shuffle(buffer_size=5,seed=42,reshuffle_each_iteration=False).batch(7)
for item in dataset:
    print(item)

tf.Tensor([2 5 1 7 0 8 9], shape=(7,), dtype=int64)
tf.Tensor([0 4 3 1 5 6 4], shape=(7,), dtype=int64)
tf.Tensor([6 7 2 9 0 3 3], shape=(7,), dtype=int64)
tf.Tensor([2 8 4 6 8 9 7], shape=(7,), dtype=int64)
tf.Tensor([5 1], shape=(2,), dtype=int64)


交织来自多个文件的行

In [65]:
from sklearn.datasets import fetch_california_housing

# 加载数据集
housing = fetch_california_housing()

# 特征 X + 标签 y
X = housing.data       # 输入特征 (8个维度)
y = housing.target     # 房价标签

In [66]:
import os
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import tensorflow as tf

# ---------------------- 1. 加载&预处理数据 ----------------------
housing = fetch_california_housing()
X, y = housing.data, housing.target
feature_names = housing.feature_names  # 8个特征名

# 合并特征+标签，生成DataFrame
df = pd.DataFrame(X, columns=feature_names)
df["MedianHouseValue"] = y  # 目标列

# 乱序+划分训练/验证/测试集（书中要求）
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
train_df, temp_df = train_test_split(df_shuffled, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# ---------------------- 2. 创建文件夹 ----------------------
os.makedirs("datasets/housing", exist_ok=True)

# ---------------------- 3. 拆分多CSV文件（模拟书中场景） ----------------------
# 训练集拆分为10个CSV文件
n_train_files = 10
train_chunks = np.array_split(train_df, n_train_files)
train_filepaths = []
for i, chunk in enumerate(train_chunks):
    filepath = f"datasets/housing/my_train_{i:02d}.csv"
    chunk.to_csv(filepath, index=False)
    train_filepaths.append(filepath)

# 验证集拆分为2个CSV文件
n_val_files = 2
val_chunks = np.array_split(val_df, n_val_files)
val_filepaths = []
for i, chunk in enumerate(val_chunks):
    filepath = f"datasets/housing/my_val_{i:02d}.csv"
    chunk.to_csv(filepath, index=False)
    val_filepaths.append(filepath)

# 测试集拆分为2个CSV文件
n_test_files = 2
test_chunks = np.array_split(test_df, n_test_files)
test_filepaths = []
for i, chunk in enumerate(test_chunks):
    filepath = f"datasets/housing/my_test_{i:02d}.csv"
    chunk.to_csv(filepath, index=False)
    test_filepaths.append(filepath)

# 打印训练文件路径
print("训练文件路径列表：")
print(train_filepaths)

训练文件路径列表：
['datasets/housing/my_train_00.csv', 'datasets/housing/my_train_01.csv', 'datasets/housing/my_train_02.csv', 'datasets/housing/my_train_03.csv', 'datasets/housing/my_train_04.csv', 'datasets/housing/my_train_05.csv', 'datasets/housing/my_train_06.csv', 'datasets/housing/my_train_07.csv', 'datasets/housing/my_train_08.csv', 'datasets/housing/my_train_09.csv']


C:\Users\24677\PycharmProjects\Hands-On_Machine_Learning_with_Scikit_Learn&TensorFlow\.venv\lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [67]:
train_filepaths

['datasets/housing/my_train_00.csv',
 'datasets/housing/my_train_01.csv',
 'datasets/housing/my_train_02.csv',
 'datasets/housing/my_train_03.csv',
 'datasets/housing/my_train_04.csv',
 'datasets/housing/my_train_05.csv',
 'datasets/housing/my_train_06.csv',
 'datasets/housing/my_train_07.csv',
 'datasets/housing/my_train_08.csv',
 'datasets/housing/my_train_09.csv']

In [68]:
# 创建一个包含以下文件路径的数据集
filepath_dataset = tf.data.Dataset.list_files(train_filepaths,seed=42)

In [69]:
# 一次读取五个文件并交织他们的行，用skip方法跳过每个文件的第一行，标题行
n_readers = 5
dataset = filepath_dataset.interleave(
    lambda filepath:tf.data.TextLineDataset(filepath).skip(1),
    cycle_length=n_readers,
)

为了交织效果更好，最好使用具有相同长度的文件，否则最长文件的结尾将不会交织

In [70]:
for line in dataset.take(3):
    print(line.numpy())

b'5.1664,32.0,5.825454545454545,1.1527272727272728,752.0,2.7345454545454544,37.16,-122.12,1.851'
b'4.2188,26.0,6.534883720930233,0.9488372093023256,627.0,2.916279069767442,39.12,-121.62,0.942'
b'3.1857,24.0,4.90249433106576,1.0408163265306123,1468.0,3.328798185941043,37.67,-121.03,0.983'


这些是随机文件的第一行（非标题行）

## 13.1.3 预处理数据

In [71]:
# 执行预处理的小函数
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. 加载数据并划分训练集
housing = fetch_california_housing()
X_train, X_test, y_train, y_test = train_test_split(
    housing.data, housing.target, test_size=0.2, random_state=42
)

# 2. 计算训练集的均值和标准差
scaler = StandardScaler()
scaler.fit(X_train)  # 仅用训练集拟合，避免数据泄露
X_mean = scaler.mean_       # shape=(8,)，8个特征的均值
X_std = scaler.scale_       # shape=(8,)，8个特征的标准差
n_inputs = 8  # 特征数量，固定为8
def preprocess(line):
    defs = [0.]*n_inputs+[tf.constant([],dtype=tf.float32)]
    fields = tf.io.decode_csv(line,record_defaults=defs)
    x = tf.stack(fields[:-1])
    y = tf.stack(fields[-1:])
    return (x-X_mean)/X_std,y


---

## 一、核心前提
代码基于**加州住房数据集**，有 8 个输入特征 + 1 个目标房价，预处理的核心目标是：
> 把 CSV 文本行 → 解析成张量 → 标准化特征 → 输出 `(特征, 标签)` 元组，用于模型训练


---

## 二、逐点拆解书中的核心说明
### 1. 预处理的前置条件
- 必须**预先用训练集计算**每个特征的均值 `X_mean` 和标准差 `X_std`（绝对不能用测试集，避免数据泄露）
- `X_mean`、`X_std` 都是**一维张量/NumPy数组**，长度为 8，和 8 个输入特征一一对应

---

### 2. `tf.io.decode_csv()` 函数详解
这是解析 CSV 的核心 API，两个关键参数：
| 参数 | 作用 |
|------|------|
| `line` | 输入：CSV 的一行文本（字符串张量） |
| `record_defaults` | 核心配置：**每列的默认值数组**，同时告诉 TensorFlow 3 个关键信息：<br>① 列数（数组长度=列数）<br>② 每列的数据类型（默认值的类型）<br>③ 缺失值的处理方式 |

#### 本案例的 `record_defaults` 设计

- 特征列允许缺失值，用 0 填充；目标列不允许缺失值，缺失直接报错，保证标签完整性

---

### 3. `tf.stack()` 的作用
- `tf.io.decode_csv()` 返回的是**9个标量张量的列表**（每列一个标量）
- 我们需要把 8 个特征标量，转换成**一维特征张量**，用于模型输入
- `tf.stack(fields[:8])` 就是把 8 个标量堆叠成 shape=(8,) 的一维张量，符合神经网络的输入格式
- 目标值 `y` 保留标量（或也可堆叠成 shape=(1,) 的一维张量，根据模型需求调整）

---

### 4. 特征标准化（Z-Score 标准化）
```python
x = (x - X_mean) / X_std
```
- 公式：$x_{\text{标准化}} = \frac{x - \mu}{\sigma}$
- 作用：把每个特征缩放到**均值为0、标准差为1**，让神经网络训练更稳定、收敛更快
- 核心原则：**仅用训练集的均值/标准差**，验证集、测试集必须复用训练集的统计量，绝对不能重新计算

---

### 5. 最终输出
函数返回一个元组 `(x, y)`：
- `x`：标准化后的一维特征张量（shape=(8,)）
- `y`：目标房价标量张量（或一维张量）
- 直接适配 `tf.data.Dataset.map(preprocess)`，可无缝接入数据 pipeline，用于模型训练

---

## 三、核心知识点总结（记忆版）
| 步骤 | 核心操作 | 关键作用 |
|------|----------|----------|
| 1. 前置准备 | 计算训练集的 `X_mean`、`X_std` | 标准化的统计依据，避免数据泄露 |
| 2. 定义默认值 | 配置 `record_defaults` | 告诉TF列数、类型、缺失值处理规则 |
| 3. 解析CSV | `tf.io.decode_csv()` | 把文本行转换成标量张量列表 |
| 4. 拆分堆叠 | `tf.stack(fields[:8])` | 把标量列表转换成一维特征张量 |
| 5. 标准化 | `(x - X_mean) / X_std` | 特征缩放，加速模型训练 |
| 6. 输出 | 返回 `(x, y)` 元组 | 适配模型输入，接入训练 pipeline |

---

## 四、关键避坑指南
1.  **绝对禁止用测试集计算均值/标准差**：这是数据泄露的典型错误，会导致模型评估严重失真
2.  **`record_defaults` 长度必须和CSV列数一致**：否则会解析报错
3.  **目标列的默认值设置**：用空数组保证标签无缺失，避免模型学到错误的标签
4.  **`tf.stack` 仅用于特征列**：目标列保留标量，无需堆叠（除非模型要求一维输入）
5.  **预处理函数必须在 `map` 中调用**：在 `tf.data` pipeline 中实现流式预处理，支持大规模数据

---

## 五、一句话总结
`preprocess` 函数的核心是**把CSV文本行解析成张量、拆分特征标签、标准化特征**，最终输出模型可用的 `(特征, 标签)` 元组，是TensorFlow处理大规模CSV数据的标准预处理模板。


In [72]:
# 测试一下预处理函数
preprocess(b'4.2188,26.0,6.534883720930233,0.9488372093023256,627.0,2.916279069767442,39.12,-121.62,0.942')

(<tf.Tensor: shape=(8,), dtype=float32, numpy=
 array([ 0.17752305, -0.20697188,  0.46062383, -0.34129035, -0.7031113 ,
        -0.01560512,  1.6272806 , -1.0160148 ], dtype=float32)>,
 <tf.Tensor: shape=(1,), dtype=float32, numpy=array([0.942], dtype=float32)>)

## 13.1.4 合并在一起

In [73]:
# 将之前的所有内容放到一个小的辅助函数中：创建并返回一个数据集，该数据集可以有效地从多个CSV文件中加载加州住房数据，对其进行预处理，随机乱序，可以选择重复，并进行批处理
def csv_reader_dataset(filepaths,repeat = 1,n_readers = 5,n_reader_threads = None,shuffle_buffer_size = 10000,n_parse_threads = 5,batch_size = 32):
    dataset = tf.data.Dataset.list_files(filepaths)
    dataset = dataset.interleave(
        lambda filepath:tf.data.TextLineDataset(filepath).skip(1),
        cycle_length=n_readers,num_parallel_calls=n_reader_threads
    )
    dataset = dataset.map(preprocess,num_parallel_calls=n_parse_threads)
    dataset = dataset.shuffle(shuffle_buffer_size).repeat(repeat)
    return dataset.batch(batch_size).prefetch(1)


---

## 一、函数整体定位
这是一个**生产级的 tf.data 数据管道封装函数**，把「多 CSV 文件读取 → 交错去表头 → 并行解析预处理 → 乱序重复 → 分批预取」全流程打包，直接输出可用于模型训练的数据集，是 TensorFlow 处理大规模 CSV 数据的标准模板。

---

## 二、第一步：函数参数全解析
```python
def csv_reader_dataset(filepaths, repeat = 1,n_readers = 5,n_reader_threads = None,shuffle_buffer_size = 10000,n_parse_threads = 5,batch_size = 32):
```
| 参数 | 含义 | 核心作用 | 调优建议 |
|------|------|----------|----------|
| `filepaths` | CSV 文件路径列表 | 输入：所有训练/验证/测试 CSV 文件的路径 | 支持通配符（如 `datasets/housing/my_train_*.csv`） |
| `repeat=1` | 数据集重复次数 | 控制数据集遍历轮数：`1`=遍历1次，`None`=无限重复（训练用） | 训练时设 `repeat=None`，配合 `steps_per_epoch` 使用 |
| `n_readers=5` | 同时读取的文件数（`cycle_length`） | 控制 `interleave` 并行度，同时打开5个文件交错读取 | 一般设为 CPU 核心数或文件数，不建议超过10 |
| `n_reader_threads=None` | 文件读取并行线程数 | `interleave` 的并行读取线程，`None`=自动优化 | 生产环境设 `tf.data.AUTOTUNE`，让 TF 自动调优 |
| `shuffle_buffer_size=10000` | 乱序缓冲区大小 | 控制数据乱序的彻底性：值越大，乱序越均匀，越耗内存 | 训练集设为样本数的10%~100%，验证/测试集可设为1（不打乱） |
| `n_parse_threads=5` | 解析并行线程数 | `map` 预处理的并行线程数 | 设为 CPU 核心数，或 `tf.data.AUTOTUNE` |
| `batch_size=32` | 批次大小 | 每批喂给模型的样本数 | 常见值：16/32/64/128，根据显存调整 |

---

## 三、第二步：逐行代码执行流程拆解
### 1. 生成文件路径数据集
```python
dataset = tf.data.Dataset.list_files(filepaths)
```
- **作用**：把输入的文件路径列表，转换成一个**元素为文件路径的数据集**
- **关键特性**：默认自动乱序文件路径（`shuffle=True`），避免后续按文件顺序读取，保证数据随机性
- **示例输出**：`["my_train_02.csv", "my_train_00.csv", "my_train_05.csv", ...]`（乱序的文件路径）

---

### 2. 多文件交错读取 + 跳过表头（核心 IO 优化）
```python
dataset = dataset.interleave(
    lambda filepath:tf.data.TextLineDataset(filepath).skip(1),
    cycle_length=n_readers,
    num_parallel_calls=n_reader_threads
)
```
#### 逐部分拆解
| 代码 | 作用 |
|------|------|
| `tf.data.TextLineDataset(filepath)` | 把单个 CSV 文件转换成**按行读取的文本数据集**，每个元素是一行CSV文本（字符串张量），流式读取，不占内存 |
| `.skip(1)` | 跳过每个文件的**第1行（表头）**，只保留数据行，避免把表头当成训练数据 |
| `lambda filepath: ...` | 匿名函数，把「文件路径 → 去表头的行数据集」的逻辑封装，传给 `interleave` |
| `cycle_length=n_readers=5` | 同时打开5个文件，从每个文件依次读取一行，交错合并行数据 |
| `num_parallel_calls=n_reader_threads` | 多线程并行读取文件，大幅提升IO速度，避免IO瓶颈 |

#### 交错读取的效果（举个例子）
5个文件A/B/C/D/E，表头已跳过：
- 读取顺序：A1 → B1 → C1 → D1 → E1 → A2 → B2 → C2 → D2 → E2 → ...
- 最终效果：完全打乱文件间的顺序，避免模型按文件顺序学习，提升泛化性

---

### 3. 并行解析 + 预处理
```python
dataset = dataset.map(preprocess, num_parallel_calls=n_parse_threads)
```
- **作用**：对数据集中的每一行CSV文本，执行 `preprocess` 预处理函数
- **`preprocess` 函数的核心逻辑**（之前标准实现）：
  1.  用 `tf.io.decode_csv` 解析文本行，拆分8个特征 + 1个标签
  2.  用训练集的 `X_mean`/`X_std` 对特征做标准化（`(x - mean)/std`）
  3.  返回 `(特征张量, 标签张量)` 元组，直接适配模型输入
- **`num_parallel_calls=n_parse_threads=5`**：多线程并行执行预处理，充分利用CPU，避免解析成为瓶颈
- **生产环境建议**：设为 `tf.data.AUTOTUNE`，让 TensorFlow 自动优化线程数

---

### 4. 乱序 + 重复（训练核心步骤）
```python
dataset = dataset.shuffle(shuffle_buffer_size).repeat(repeat)
```
#### 顺序逻辑：**先乱序，后重复**（绝对不能反过来！）
| 操作 | 作用 |
|------|------|
| `shuffle(shuffle_buffer_size=10000)` | 用大小为10000的缓冲区随机打乱数据：每次从缓冲区随机取一个元素，再用新元素填充，保证数据随机性 |
| `repeat(repeat=1)` | 重复整个数据集：`1`=遍历1次，`None`=无限重复（训练时用，配合 `steps_per_epoch`） |

#### 为什么不能先重复后乱序？
- 先重复后乱序：会导致每个 epoch 内数据乱序，但 epoch 之间顺序重复，模型会记住 epoch 顺序，过拟合
- 先乱序后重复：每个 epoch 都是全新的乱序，模型永远学不到顺序，泛化性最好

---

### 5. 分批 + 预取（性能优化核心）
```python
return dataset.batch(batch_size).prefetch(1)
```
| 操作 | 作用 |
|------|------|
| `batch(batch_size=32)` | 把连续的样本打包成批次，输出形状：`(batch_size, 8)` 特征 + `(batch_size,)` 标签，适配模型输入 |
| `prefetch(1)` | 异步预取下一个批次的数据：模型训练第N批时，CPU提前加载好第N+1批，消除GPU等待时间，最大化GPU利用率 |

#### 生产环境优化
- `prefetch(1)` 可以替换为 `prefetch(tf.data.AUTOTUNE)`，让 TensorFlow 自动优化预取批次数量，性能更好
- 若需要丢弃最后一个不足 `batch_size` 的批次，可加 `drop_remainder=True`：`batch(batch_size, drop_remainder=True)`

---

## 四、完整执行流程（从文件到模型输入）
```
CSV文件列表 → list_files（乱序文件路径） → interleave（交错读取+去表头） → map（并行解析+标准化） → shuffle（乱序样本） → repeat（重复数据集） → batch（分批） → prefetch（预取） → 模型训练
```

---

## 五、关键避坑指南
1.  **`skip(1)` 必须加**：CSV表头必须跳过，否则会把表头字符串当成数据，导致类型错误
2.  **乱序顺序绝对不能反**：必须 `shuffle` 在前，`repeat` 在后，否则会导致 epoch 顺序重复
3.  **`num_parallel_calls` 一定要加**：无论是 `interleave` 还是 `map`，都要开并行，否则IO/解析会成为瓶颈
4.  **`prefetch` 不能漏**：这是消除GPU等待时间的核心，训练时必须加
5.  **预处理统计量只能用训练集**：`X_mean`/`X_std` 必须用训练集计算，验证/测试集复用，绝对不能用测试集，避免数据泄露

---

## 六、最终输出说明
函数返回一个 `PrefetchDataset`，是完全就绪的训练数据集：
- 每个元素是 `(特征批次, 标签批次)` 元组
- 特征批次形状：`(batch_size, 8)`（32个样本，8个标准化特征）
- 标签批次形状：`(batch_size,)`（32个房价标签）
- 直接可以传给 `model.fit(dataset)`，无需任何额外处理

---

## 七、一句话总结
这个函数是**CSV 版大规模数据加载的最佳实践**，通过 `interleave` 并行交错读取保证数据随机性，`map` 并行解析预处理，`shuffle`/`repeat`/`batch`/`prefetch` 构建高性能训练 pipeline，完美适配加州住房数据集，可直接复用在所有 CSV 回归/分类任务中。

---


## 13.1.5 预取


这一节是 TensorFlow 数据管道优化的**核心章节**，核心讲的是**如何利用多线程并行，彻底消除数据加载等待，最大化 GPU 利用率**。

---

## 一、核心概念：什么是预取（Prefetch）？
### 1. 定义
`prefetch(1)` 的作用是：**让数据集在后台（CPU）异步准备好下一个批次的数据**，而不是等 GPU 用完当前批次再去加载。

### 2. 一句话原理
> CPU 负责准备数据（读取+预处理），GPU 负责训练模型。两者**并行工作**，实现“流水线”式流转，避免 GPU 空等数据加载。

---

## 二、为什么预取能提升性能？
### 1. 没有预取（串行工作）❌
- 流程：GPU 训练完当前 batch → CPU 加载下一个 batch → GPU 开始训练
- 结果：**GPU 经常处于等待状态**，算力利用率低，训练慢。

### 2. 有预取（并行工作）✅
- 流程：GPU 训练 batch N 时 → CPU 异步提前加载并预处理 batch N+1
- 结果：**GPU 几乎 100% 利用**，数据无缝切换，训练速度极快。

### 3. 有预取+多线程（极致优化）🚀
- 利用 `interleave` 和 `map` 的多线程，CPU 同时准备多个批次。
- 结果：数据准备速度 > GPU训练速度，训练效率拉满。

---

## 三、代码核心逻辑
### 关键代码行
```python
return dataset.batch(batch_size).prefetch(1)
```
### 1. `batch(batch_size)`
- 把流式数据打包成模型需要的 `(batch_size, 8)` 张量。
- 这是喂给 GPU 的标准格式。

### 2. `prefetch(1)`（核心调优点）
- **参数 `1`**：表示 CPU 提前准备好 **1 个批次** 给 GPU。
- **作用**：
  - 模型正在处理第 N 批数据时。
  - CPU 已经异步处理好第 N+1 批数据。
  - GPU 处理完第 N 批后，直接无缝读取第 N+1 批，无等待。

### 3. 生产环境最佳实践
```python
# 推荐写法：自动决定预取数量（AUTOTUNE）
dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
```
- `tf.data.AUTOTUNE`：让 TensorFlow 自动根据系统负载和 GPU 速度，动态调整预取的批次数量，是目前最推荐的方式。

---

## 四、书中提到的扩展技巧（重要补充）
### 1. 多级预取与多线程加载
书中提到了图 13-3 中的对比，要获得好性能必须设置：
- `interleave(..., num_parallel_calls=...)`：多线程并发读取多个文件。
- `map(..., num_parallel_calls=...)`：多线程并行执行 `preprocess` 解析。
**只有配合这两个参数，`prefetch` 才能发挥最大作用**，否则 CPU 会处理不过来。

### 2. `cache()` 缓存（针对小数据集）
如果数据集足够小，能放进内存：
```python
dataset = dataset.cache()  # 缓存到内存（或文件），只读取预处理一次
```
- **作用**：每个实例只被读取和预处理一次（第一轮），后续轮次直接从内存读取，极快。
- **注意**：必须在 `shuffle`、`repeat`、`batch`、`prefetch` **之前** 调用 `cache()`。
- **原理**：避免了每次 `repeat` 都重新遍历文件，是加速小数据集训练的神器。

### 3. 直接预取到 GPU
书中注释提到：
```python
tf.data.experimental.prefetch_to_device('/device:GPU:0')
```
- **作用**：直接将数据预取到 GPU 显存，跳过 CPU 到 GPU 的传输拷贝步骤。
- **适用场景**：大规模数据集训练，极致优化 IO 延迟（目前是实验性功能，需查看最新 API）。

---

## 五、其他常见数据集方法
书中提到了除了 `map`、`shuffle`、`batch` 之外的常用 API：
| API | 作用 | 场景 |
| :--- | :--- | :--- |
| `concatenate()` | 拼接两个数据集 | 合并多部分数据 |
| `zip()` | 压缩多个数据集 | 多输入/多输出模型配对 |
| `window()` | 创建窗口数据集 | 时间序列处理 |
| `shard()` | 数据分片 | 多设备并行训练（Sharded Datasets） |
| `flat_map()` | 扁平化映射 | 展平嵌套数据集 |
| `from_generator()` | 从Python生成器创建 | 支持复杂的Python生成逻辑 |

---

## 六、核心知识点总结
1. **`prefetch(1)` 是性能核心**：它是实现 CPU/GPU 并行的关键，消除数据加载的等待时间。
2. **顺序至关重要**：
   - 正确顺序：`shuffle` → `repeat` → `batch` → `prefetch`
   - 错误顺序：`batch` 过早会导致乱序不彻底
3. **多线程必须开启**：`interleave` 和 `map` 必须设置 `num_parallel_calls`，否则会成为瓶颈。
4. **`cache()` 缓存小数据**：能放入内存的数据集，先 `cache()` 再处理，极大提升速度。
5. **目标**：让 CPU 准备数据的速度 >= GPU 训练数据的速度，实现 GPU 100% 利用率。

---

## 七、一句话总结
13.1.5 节核心讲的是**预取（Prefetch）**：通过 `prefetch(1)` 让 CPU 和 GPU 并行工作，CPU 异步准备下一批数据，让 GPU 永远不等待，从而**最大化 GPU 利用率，大幅提升训练速度**，是 TensorFlow 数据管道优化的基石。

---


## 13.1.6 和tf.keras一起使用数据集

In [74]:
# 为测试集和验证集创建数据集
train_set = csv_reader_dataset(train_filepaths)
val_set = csv_reader_dataset(val_filepaths)
test_set = csv_reader_dataset(test_filepaths)

In [75]:
# 简单的构建keras模型,将训练数据集和测试数据集传送给fit（）方法，而不是X_train,y_train,x_valid和y_valid
import keras
model = keras.models.Sequential([
    keras.layers.Dense(64, activation=tf.nn.relu, input_shape=(n_inputs,)),
    keras.layers.Dense(64, activation=tf.nn.relu),
    keras.layers.Dense(64, activation='softmax')
])
model.compile(
    loss = 'sparse_categorical_crossentropy',
    optimizer = tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    metrics = ['accuracy']
)
model.fit(train_set, epochs=10, validation_data=val_set)

Epoch 1/10
452/452 [==============================] - 1s 2ms/step - loss: 1.2912 - accuracy: 0.0025 - val_loss: 1.0117 - val_accuracy: 0.0026
Epoch 2/10
452/452 [==============================] - 1s 2ms/step - loss: 1.0055 - accuracy: 0.0030 - val_loss: 0.9785 - val_accuracy: 0.0026
Epoch 3/10
452/452 [==============================] - 1s 2ms/step - loss: 0.9674 - accuracy: 0.0030 - val_loss: 0.9392 - val_accuracy: 0.0023
Epoch 4/10
452/452 [==============================] - 1s 2ms/step - loss: 0.9549 - accuracy: 0.0028 - val_loss: 0.9379 - val_accuracy: 0.0029
Epoch 5/10
452/452 [==============================] - 1s 2ms/step - loss: 0.9287 - accuracy: 0.0030 - val_loss: 0.9058 - val_accuracy: 0.0019
Epoch 6/10
452/452 [==============================] - 1s 2ms/step - loss: 0.9097 - accuracy: 0.0033 - val_loss: 0.8860 - val_accuracy: 0.0026
Epoch 7/10
452/452 [==============================] - 1s 2ms/step - loss: 0.8968 - accuracy: 0.0030 - val_loss: 0.8693 - val_accuracy: 0.0023
Epoch 

In [76]:
#将数据集传递给valuate()方法和predict()方法
model.evaluate(test_set)


97/97 [==============================] - 0s 1ms/step - loss: 0.9000 - accuracy: 0.0023


[0.8999641537666321, 0.0022609818261116743]

In [77]:
new_set = test_set.take(3).map(lambda X,y:X)# pretend we have 3 new instances
model.predict(new_set)

3/3 [==============================] - 0s 4ms/step


array([[1.62583962e-03, 1.10819981e-01, 6.42218769e-01, ...,
        6.62264938e-05, 7.39552343e-05, 4.25209109e-05],
       [8.81331787e-02, 8.84375930e-01, 2.62894463e-02, ...,
        1.93515939e-06, 1.78909170e-06, 1.84875864e-06],
       [5.10625504e-02, 7.99801528e-01, 1.30717456e-01, ...,
        3.87939326e-05, 3.99731980e-05, 2.99308922e-05],
       ...,
       [6.56249106e-01, 3.35683376e-01, 7.11311586e-03, ...,
        4.08143751e-06, 3.19471269e-06, 2.41645694e-06],
       [9.61979985e-01, 3.78880873e-02, 1.15862094e-04, ...,
        2.37606756e-07, 2.46768394e-07, 1.30079343e-07],
       [6.75182854e-10, 1.41941754e-07, 6.52938252e-05, ...,
        2.57931609e-09, 1.07213323e-08, 1.34060174e-09]], dtype=float32)

new_set 通常不包含标签

In [78]:
# 构建自定义训练循环
#  定义损失函数 loss_fn
loss_fn = tf.keras.losses.MeanSquaredError()

# 定义优化器 optimizer
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

for X_batch, y_batch in train_set:
    with tf.GradientTape() as tape:
        y_pred = model(X_batch, training=True)
        main_loss = tf.reduce_mean(loss_fn(y_batch, y_pred))
        loss = tf.add_n([main_loss] + model.losses)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))


In [79]:
@tf.function
def train(model, optimizer, loss_fn, n_epochs, train_filepaths):
    # 创建训练数据集
    train_set = csv_reader_dataset(train_filepaths, repeat=n_epochs)

    # 训练循环主体
    for X_batch, y_batch in train_set:
        with tf.GradientTape() as tape:
            # 前向传播
            y_pred = model(X_batch)
            # 计算损失（加上模型自身的损失，如正则化）
            main_loss = tf.reduce_mean(loss_fn(y_batch, y_pred))
            loss = tf.add_n([main_loss] + model.losses)
        # 反向传播更新权重
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

# 13.2 TFRecord格式

TFRecord格式是TensorFlow首选的格式，用于存储大量数据并有效读取数据。这是一种非常简单的二进制格式，只包含大小不同的二进制大小序列

In [80]:
# 创建TFRecord文件：
with tf.io.TFRecordWriter('my_data.tfrecord') as f:
    f.write(b'this is the first record')
    f.write(b'this is the second record')

In [81]:
# 读取一个或多个TFRecord文件
filepaths = ['my_data.tfrecord']
dataset = tf.data.TFRecordDataset(filepaths)
for item in dataset:
    print(item)

tf.Tensor(b'this is the first record', shape=(), dtype=string)
tf.Tensor(b'this is the second record', shape=(), dtype=string)


默认情况下，TFRecordDataset将一个接一个地读取文件，但是可以通过设置num_parallel_reads使其并行读取多个文件并交织结果

## 13.2.1 压缩的TFRecord文件

In [82]:
import tensorflow as tf

# 1. 定义要写入的内容
my_example_serialized = b"this is the first compressed record"

# 2. 写入压缩的 TFRecord 文件
options = tf.io.TFRecordOptions(compression_type="GZIP")
with tf.io.TFRecordWriter("my_compressed.tfrecord", options) as f:
    f.write(my_example_serialized)

# 3. 读取验证（需要指定压缩类型）
dataset = tf.data.TFRecordDataset("my_compressed.tfrecord", compression_type="GZIP")
for item in dataset:
    print(item)

tf.Tensor(b'this is the first compressed record', shape=(), dtype=string)


## 13.2.2 协议缓冲区简介

In [83]:
from person_pb2 import Person

person = Person(name="Al", id=123, email=["a@b.com"])
print(person)

name: "Al"
id: 123
email: "a@b.com"



In [84]:
person.name

'Al'

In [85]:
person.email[0]

'a@b.com'

In [86]:
person.email.append('c@d.com')
s = person.SerializeToString()
s

b'\n\x02Al\x10{\x1a\x07a@b.com\x1a\x07c@d.com'

In [87]:
person2 = Person()
person2.ParseFromString(s)

24

In [88]:
person == person2

True

## 13.2.3 TensorFlow协议

In [89]:
# 1. 导入 TensorFlow 自带的 Protobuf 类
from tensorflow.train import BytesList, FloatList, Int64List
from tensorflow.train import Feature, Features, Example

# 2. 构建和书上完全一样的 Person Example
person_example = Example(
    features=Features(
        feature={
            "name": Feature(bytes_list=BytesList(value=[b"Alice"])),
            "id": Feature(int64_list=Int64List(value=[123])),
            "emails": Feature(bytes_list=BytesList(value=[b"a@b.com", b"c@d.com"]))
        }
    )
)

# 3. 写入 TFRecord 文件
with tf.io.TFRecordWriter("my_contacts.tfrecord") as f:
    f.write(person_example.SerializeToString())

print("✅ 运行成功！已生成 my_contacts.tfrecord 文件")

✅ 运行成功！已生成 my_contacts.tfrecord 文件


## 13.2.4 加载和解析Example

In [90]:
# 定义一个描述字典，然后遍历TFRecord Dataset并解析该数据集包含的序列化的Example protobuf
feature_description = {
    'name':tf.io.FixedLenFeature([], tf.string,default_value=""),
    'id':tf.io.FixedLenFeature([], tf.int64,default_value=-0),
    'emails':tf.io.VarLenFeature(tf.string),
}
for serialized_example in tf.data.TFRecordDataset(['my_contacts.tfrecord']):
    parsed_example = tf.io.parse_single_example(serialized_example, feature_description)

In [91]:
# 将稀疏张量转换为规则张量
tf.sparse.to_dense(parsed_example['emails'], default_value=b"")


<tf.Tensor: shape=(2,), dtype=string, numpy=array([b'a@b.com', b'c@d.com'], dtype=object)>

In [92]:
parsed_example['emails'].values

<tf.Tensor: shape=(2,), dtype=string, numpy=array([b'a@b.com', b'c@d.com'], dtype=object)>

In [93]:
#一个批次接一个批次的解析
dataset = tf.data.TFRecordDataset(['my_contacts.tfrecord']).batch(10)
for serialized_examples in dataset:
    parsed_examples = tf.io.parse_example(serialized_examples, feature_description)

## 13.2.5 使用SequenceExample Protobuf处理列表的列表

In [94]:
import tensorflow as tf

# ======================
# 1. 先定义特征描述
# ======================

context_feature_description = {
    "label": tf.io.FixedLenFeature([], tf.int64, default_value=0),
    "author": tf.io.FixedLenFeature([], tf.string, default_value="unknown")
}

# 序列特征
sequence_feature_description = {
    "content": tf.io.VarLenFeature(tf.int64),
    "comments": tf.io.VarLenFeature(tf.string)
}

# ======================
# 2. 创建一个 SequenceExample
# ======================
# 上下文特征
context = tf.train.Features(feature={
    "label": tf.train.Feature(int64_list=tf.train.Int64List(value=[1])),
    "author": tf.train.Feature(bytes_list=tf.train.BytesList(value=[b"John Doe"]))
})

# 序列特征（句子列表：每个句子是单词ID列表）
feature_lists = tf.train.FeatureLists(feature_list={
    "content": tf.train.FeatureList(feature=[
        tf.train.Feature(int64_list=tf.train.Int64List(value=[1, 2, 3])),
        tf.train.Feature(int64_list=tf.train.Int64List(value=[4, 5])),
        tf.train.Feature(int64_list=tf.train.Int64List(value=[6, 7, 8, 9]))
    ]),
    "comments": tf.train.FeatureList(feature=[
        tf.train.Feature(bytes_list=tf.train.BytesList(value=[b"good", b"example"])),
        tf.train.Feature(bytes_list=tf.train.BytesList(value=[b"nice", b"work", b"!"]))
    ])
})

# 组装成 SequenceExample
example = tf.train.SequenceExample(context=context, feature_lists=feature_lists)
serialized_example = example.SerializeToString()

# ======================
# 3. 解析
# ======================
parsed_context, parsed_feature_lists = tf.io.parse_single_sequence_example(
    serialized_example,
    context_feature_description,
    sequence_feature_description
)

# 把稀疏的序列特征转换成 RaggedTensor
parsed_content = tf.RaggedTensor.from_sparse(parsed_feature_lists['content'])

# ======================
# 4. 打印结果验证
# ======================
print("✅ 解析成功！")
print("上下文特征：")
print(f"label: {parsed_context['label'].numpy()}")
print(f"author: {parsed_context['author'].numpy().decode('utf-8')}")
print("\n序列特征 content（句子的单词ID列表）：")
print(parsed_content)

✅ 解析成功！
上下文特征：
label: 1
author: John Doe

序列特征 content（句子的单词ID列表）：
<tf.RaggedTensor [[1, 2, 3], [4, 5], [6, 7, 8, 9]]>


# 13.3 预处理输入特征

In [95]:
# 使用Lambda层实现标准化层：对于每个特征，均减去一个均值并除以标准差
means = np.mean(X_train,axis = 0,keepdims = True)
stds = np.std(X_train,axis = 0,keepdims = True)
eps = keras.models.Sequential([
    keras.layers.Lambda(lambda inputs:(inputs-means)/(stds+eps)),
    keras.layers.Dense(64, activation=tf.nn.relu),
    keras.layers.Dense(64, activation=tf.nn.relu),
])

In [96]:
# 使用一个自包含自定义层：
class Standardization(keras.layers.Layer):
    def adapt(self,data_sample):
        self.means_ = np.mean(data_sample,axis = 0,keepdims = True)
        self.stds_ = np.std(data_sample,axis = 0,keepdims = True)
    def call(self,inputs):
        return (inputs-self.means_)/(self.stds+keras.backend.epsilon())

In [97]:
# 使用此层标准化之前，需要通过调用adapt()方法来使其适应数据集并将其传递给数据样本
std_layer = Standardization()
#std_layer.adapt(data_sample)

## 13.3.1 使用独热向量编码分类特征

In [98]:
# 将每个类别映射到其索引（0到4），使用查找表来完成
vocab = ['<1H OCEAN','INLAND','NEAR OCEAN','NEAR BAY','ISLAND']
indices = tf.range(len(vocab),dtype=tf.int64)
table_init = tf.lookup.KeyValueTensorInitializer(vocab,indices)
num_oov_buckets = 2
table = tf.lookup.StaticVocabularyTable(table_init,num_oov_buckets)



这是 TensorFlow 中，**将文本类别映射为数字索引**的完整流程，共 4 步：

1.  **定义词汇表（`vocab`）**
    - 含义：列出数据中所有可能出现的类别
    - 要求：每个类别必须**唯一**，不能重复
    - 示例：`['<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'NEAR BAY']`

2.  **创建索引张量（`indices`）**
    - 含义：为词汇表中的每个类别，生成对应的数字索引（从 0 开始）
    - 示例：`tf.range(len(vocab), dtype=tf.int64)` 会生成 `[0, 1, 2, 3]`

3.  **创建初始化器（`KeyValueTensorInitializer`）**
    - 作用：把“类别-索引”的对应关系打包，供查找表使用
    - 适用场景：类别列表已经在内存中（像你现在这样）
    - 补充：如果类别存在文本文件里，需要用 `TextFileInitializer`

4.  **创建静态词汇表（`StaticVocabularyTable`）**
    - 作用：生成最终的查找表，把文本类别一键映射成数字索引
    - 关键参数 `num_oov_buckets`：词汇表外的桶数，用来处理**没见过的类别（OOV）**

---

## 核心概念：什么是 OOV 桶？
OOV = Out-of-Vocabulary，指**词汇表中不存在的类别**。

### 工作原理
- 当查找表遇到词汇表中没有的类别时，会用哈希算法把它分配到一个 OOV 桶里
- OOV 桶的索引从已知类别的最后一个索引开始，比如你有 4 个已知类别，索引是 0-3，2 个 OOV 桶的索引就是 4 和 5

### 为什么要用 OOV 桶？
- 类别数量太大、不完整、会变化时，很难收集到所有可能的类别
- 用 OOV 桶可以给未知类别分配索引，避免报错
- 桶数越多，不同未知类别冲突的概率越低（冲突=不同类别被分到同一个桶，模型无法区分）

---

## 关键避坑点
1.  **词汇表必须唯一**：重复类别会导致同一个键对应多个值，直接报错
2.  **索引类型必须是 `tf.int64`**：`StaticVocabularyTable` 要求索引为 `int64` 类型
3.  **OOV 桶数要合理**：太少会导致冲突，太多会浪费内存，一般根据未知类别的数量预估

---

In [99]:
categories = tf.constant(['NEAR BAY','DESERT','INLAND','INLAND'])
cat_indices = table.lookup(categories)
cat_indices

<tf.Tensor: shape=(4,), dtype=int64, numpy=array([3, 5, 1, 1], dtype=int64)>

| 输入类别 | 词表中是否存在？ | 映射后的索引 | 独热编码结果 |
|----------|----------------|--------------|----------------------|
| NEAR BAY | ✅ 存在 | 3 | [0, 0, 0, 1, 0, 0, 0] |
| DESERT   | ❌ 不存在 | 5 | [0, 0, 0, 0, 0, 1, 0] |
| INLAND   | ✅ 存在 | 1 | [0, 1, 0, 0, 0, 0, 0] |
| INLAND   | ✅ 存在 | 1 | [0, 1, 0, 0, 0, 0, 0] |

In [100]:
cat_one_hot = tf.one_hot(cat_indices,depth=len(vocab)+num_oov_buckets)
cat_one_hot

<tf.Tensor: shape=(4, 7), dtype=float32, numpy=
array([[0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0.],
       [0., 1., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0.]], dtype=float32)>

## 13.3.2 使用嵌入编码分类特征


---

## 一、核心主题：用嵌入（Embedding）编码分类特征
这部分是讲如何用**嵌入向量**替代独热编码，来处理分类特征，是深度学习中非常重要的技巧。

### 1. 什么是嵌入？
- 嵌入是**可训练的密集向量**，用来表示分类特征（比如类别、单词）。
- 它是随机初始化的（比如“NEAR BAY”一开始是 `[0.131, 0.890]` 这样的随机向量），但会在训练过程中不断优化。
- 训练的目标：**让相似的类别在向量空间中距离更近**。比如“NEAR OCEAN”和“NEAR BAY”的向量会越来越接近，而和“INLAND”的向量会越来越远。

### 2. 嵌入 vs 独热编码：怎么选？
书上给了一个实用的经验规则：
| 类别数量 | 推荐方法 | 原因 |
| :--- | :--- | :--- |
| 很少（≤10个） | 独热编码 | 简单直接，效果足够好 |
| 很多（>50个） | 嵌入 | 独热编码会产生高维稀疏向量，效率低；嵌入是低维密集向量，更高效 |
| 10-50个 | 两种都试试 | 看哪个效果更好 |

---

## 二、嵌入的核心优势：捕捉语义关系
嵌入最强大的地方，就是能自动捕捉类别之间的语义关系，书上举了两个非常经典的例子：

### 1. 地理类别的嵌入（你的书上第一个图）
- 模型会自动学到：“NEAR OCEAN”和“NEAR BAY”都是靠海的，所以它们的向量会靠得很近。
- 而“INLAND”是内陆，和靠海的类别向量会离得很远。
- 这种“语义相近 → 向量相近”的特性，会让神经网络更容易做出准确预测。

### 2. 词嵌入的经典例子（你的书上第二个图）
词嵌入（Word Embedding）是NLP领域最成功的应用，它甚至能编码抽象的逻辑关系：
- `King - Man + Woman ≈ Queen`：国王（King）减去男人（Man）的向量，再加上女人（Woman）的向量，结果非常接近女王（Queen）的向量。这说明嵌入学到了“性别”这个概念。
- `Madrid - Spain + France ≈ Paris`：马德里（西班牙首都）减去西班牙，加上法国，结果接近巴黎（法国首都）。这说明嵌入学到了“国家-首都”的关系。

---

## 三、嵌入的实用价值与风险
### 1. 实用价值：可迁移性
- 预训练好的嵌入，往往可以在其他任务中重复使用。比如NLP里，用大规模语料预训练的词嵌入，比从零开始训练效果好得多。
- 这就是为什么我们常说“词嵌入是NLP的基石”。

### 2. 潜在风险：会放大数据偏见
- 嵌入会忠实地学习数据中的模式，包括偏见。比如训练数据里“男人是医生，女人是护士”的模式，会被嵌入学到，导致模型出现性别歧视。
- 这也是现在深度学习公平性研究的重点课题。

---

## 四、一句话总结
嵌入就是**让模型自己学习每个类别的“专属向量”**，既解决了独热编码的维度爆炸问题，又能自动捕捉类别间的语义关系，是处理分类特征的“进阶版”方法。

---


In [101]:
# 创建一个包含每个类别嵌入的嵌入矩阵，并随机初始化
embedding_dim = 2
embed_init = tf.random.uniform([len(vocab) + num_oov_buckets, embedding_dim])
embedding_matrix = tf.Variable(embed_init)

In [102]:
embedding_matrix

<tf.Variable 'Variable:0' shape=(7, 2) dtype=float32, numpy=
array([[0.46193063, 0.6405777 ],
       [0.6441355 , 0.64700246],
       [0.1618762 , 0.67157364],
       [0.46790516, 0.36931264],
       [0.00397968, 0.03065979],
       [0.2412715 , 0.04950726],
       [0.68563354, 0.6541407 ]], dtype=float32)>

In [103]:
# 对之前相同的分类进行嵌入编码
categories = tf.constant(['NEAR BAY','DESERT','INLAND','INLAND'])
cat_indices = table.lookup(categories)
cat_indices

<tf.Tensor: shape=(4,), dtype=int64, numpy=array([3, 5, 1, 1], dtype=int64)>

In [104]:
tf.nn.embedding_lookup(embedding_matrix,cat_indices)

<tf.Tensor: shape=(4, 2), dtype=float32, numpy=
array([[0.46790516, 0.36931264],
       [0.2412715 , 0.04950726],
       [0.6441355 , 0.64700246],
       [0.6441355 , 0.64700246]], dtype=float32)>

tf.nn.embedding_lookup()函数以给定的索引查找在嵌入矩阵中的行，这就是他所做的全部

In [105]:
#处理嵌入矩阵
embedding = keras.layers.Embedding(len(vocab) + num_oov_buckets, output_dim=embedding_dim)
embedding(cat_indices)

<tf.Tensor: shape=(4, 2), dtype=float32, numpy=
array([[ 0.01408825, -0.02411233],
       [-0.0235746 , -0.04344772],
       [-0.01637168, -0.04673366],
       [-0.01637168, -0.04673366]], dtype=float32)>

In [106]:
# 创建一个keras模型，该模型可以处理分类特征（以及常规的数值特征）并学习每个类别（以及每个oov桶）的嵌入
regular_inputs = keras.layers.Input(shape = [8])
categories = keras.layers.Input(shape = [],dtype = tf.string)
cat_indices = keras.layers.Lambda(lambda cats:table.lookup(cats),output_shape=())(categories)
cat_embed = keras.layers.Embedding(input_dim = 6,output_dim=2)(cat_indices)
encoded_inputs = keras.layers.concatenate([regular_inputs,cat_embed])
outputs = keras.layers.Dense(1)(encoded_inputs)
model = keras.models.Model(inputs=[regular_inputs,categories],outputs=[outputs])

## 13.3.3 keras预处理层


---
这一节是讲 TensorFlow 中内置的**标准预处理层**，用来简化数据预处理流程，替代旧版 Feature Columns API，让预处理更直观、更易用。

---

### 一、核心背景：为什么要用 Keras 预处理层？
- TensorFlow 团队提供了一组标准的 Keras 预处理层，未来会替代旧版 Feature Columns API
- 优势：
  1.  **直观易用**：比 Feature Columns 更符合现代 Keras 风格，代码更简洁
  2.  **可集成到模型中**：预处理层可以直接作为模型的一部分，部署时无需额外处理数据
  3.  **支持自动适配**：可以通过 `adapt()` 方法从数据中自动学习统计信息（比如均值、方差、词表）

---

### 二、已讨论过的预处理层
1.  **`keras.layers.Normalization`**
    - 功能：对数值特征执行标准化（减去均值，除以标准差）
    - 使用方式：
      - 创建层后，调用 `adapt(data_sample)` 从数据中学习均值和方差
      - 直接添加到模型中，会自动对输入数据进行标准化

2.  **`keras.layers.TextVectorization`**
    - 功能：将文本中的每个单词，编码为它在词汇表中的索引（替代之前手动写的 `Lambda + table.lookup`）
    - 使用方式：
      - 创建层后，调用 `adapt(text_data)` 自动构建词汇表
      - 直接添加到模型中，会自动把文本转换为词索引
    - 进阶用法：支持输出词袋向量（单词计数）、TF-IDF 向量等多种形式

---

### 三、其他重要预处理层
1.  **`keras.layers.Discretization`**
    - 功能：将连续数值数据切成不同的离散块（分箱），并将每个块编码为独热向量
    - 示例：将价格离散化为低、中、高三个类别，分别编码为 `[1,0,0]`、`[0,1,0]`、`[0,0,1]`
    - 注意：
      - 不可微分，参数不受梯度下降影响，因此训练过程中会冻结
      - 仅建议在模型开头使用，不建议用自定义层封装
      - 如果需要可训练的离散化，应直接使用 `Embedding` 层

2.  **`keras.layers.PreprocessingStage`**
    - 功能：链式连接多个预处理层，构建预处理流水线（类似 Scikit-Learn 的 Pipeline）
    - 示例：先对数据归一化，再离散化
    - 使用方式：
      ```python
      normalization = keras.layers.Normalization()
      discretization = keras.layers.Discretization([...])
      pipeline = keras.layers.PreprocessingStage([normalization, discretization])
      pipeline.adapt(data_sample)
      ```
    - 注意：只能在模型开头使用，因为它包含不可微分的预处理层

---

### 四、TextVectorization 的进阶用法：词袋与 TF-IDF
1.  **词袋向量（Bag-of-Words）**
    - 功能：输出每个单词在文本中的出现次数，而不是单个词索引
    - 示例：词汇表为 `["and", "basketball", "more"]`，文本 `"more and more"` 会被映射为 `[1, 0, 2]`
    - 缺点：会丢失单词顺序信息，且高频常用词（如 "and"）的权重会被放大

2.  **TF-IDF（词频-逆文档频率）**
    - 功能：对词袋向量进行归一化，降低常用词的权重，突出稀有词的重要性
    - 原理：将每个单词计数除以其出现的文本实例总数的对数
    - 示例：假设 "and" 出现 200 次、"basketball" 出现 10 次、"more" 出现 100 次，文本 `"more and more"` 的 TF-IDF 向量为 `[1/log(200), 0/log(10), 2/log(100)]`

---

### 五、自定义预处理层
如果标准预处理层无法满足需求，可以创建自定义预处理层，继承自 `keras.layers.PreprocessingLayer`，并实现：
- `adapt()` 方法：从数据中学习必要的统计信息
- `reset_state` 参数：控制是否在重新适应数据前重置状态

---

### 六、关键注意事项
- **预处理层的使用场景**：
  - 可以直接集成到模型中，部署时无需额外数据预处理
  - 训练过程中不可微分的层会被冻结，不参与梯度更新
- **预处理时机的选择**：
  - 小数据集：可以直接在训练时即时处理，用 `cache()` 方法缓存
  - 大数据集：建议在训练前用 Apache Beam/Spark 预处理，再训练模型
  - 部署时，集成在模型中的预处理层可以避免训练/服务时的处理差异问题

---


# 13.4 TF Transform


这一节讲的是 **TF Transform（TFT）**，它是解决“训练前预处理”和“部署后预处理不一致”问题的终极方案，也是TensorFlow Extended（TFX）平台的核心组件之一。

---

### 一、为什么需要 TF Transform？（痛点背景）
在大规模机器学习项目中，我们会遇到两个核心矛盾：

#### 1. 大数据集的预处理效率问题
- 预处理（比如标准化、词表构建）是计算密集型操作，在训练时即时处理会拖慢速度
- 解决方案：在训练前用 Apache Beam/Spark 对全量数据做一次预处理
- 但这样会带来新的问题：

#### 2. 训练/服务不一致（Train/Skew）
- 训练前的预处理代码（Beam/Spark），和部署时的预处理代码（App/浏览器）是两套独立的实现
- 一旦修改预处理逻辑，就要同时维护多套代码，容易出错
- 细微的差异会导致模型部署后性能下降，甚至出现错误

---

### 二、TF Transform 是什么？它的核心优势
TF Transform 是 TFX 的组件，专门解决上面的问题，核心优势是：**只定义一次预处理逻辑，同时服务于训练和部署**。

1.  **统一的预处理定义**
    - 你只需要写一个 Python 预处理函数 `preprocess(inputs)`，里面包含所有数据处理逻辑（标准化、词表构建、分箱等）
    - 它会同时生成两部分内容：
      1.  给 Apache Beam/Spark 使用的批处理代码，用于训练前对全量数据预处理
      2.  给模型使用的 TensorFlow 函数，作为预处理层直接集成到模型中，部署时自动处理输入

2.  **自动学习统计信息**
    - 它会在预处理过程中，自动计算训练集的统计信息（均值、标准差、词表、分箱边界等）
    - 这些统计信息会被打包进模型，部署时直接使用，无需重新计算

---


### 三、TF Transform 的工作流程（和书上示例对应）
1.  **定义预处理函数 `preprocess()`**
    ```python
    import tensorflow_transform as tft

    def preprocess(inputs):
        # 连续特征标准化
        standardized_age = tft.scale_to_z_score(inputs["housing_median_age"])
        # 分类特征构建词表并编码
        ocean_proximity_id = tft.compute_and_apply_vocabulary(inputs["ocean_proximity"])
        return {
            "standardized_median_age": standardized_age,
            "ocean_proximity_id": ocean_proximity_id
        }
    ```
    这一步只写一次逻辑，同时处理两种特征。

2.  **用 Apache Beam 运行预处理流水线**
    - 使用 `AnalyzeAndTransformDataset` 类，在 Apache Beam 上运行 `preprocess()` 函数
    - 对全量训练数据进行预处理，生成训练用的 TFRecord 文件
    - 同时自动计算并保存所有统计信息（均值、词表等）

3.  **生成可部署的预处理函数**
    - TF Transform 会根据统计信息，自动生成一个等效的 TensorFlow 函数
    - 你可以直接把这个函数作为预处理层，嵌入到 Keras 模型中
    - 部署时，模型会自动用同一套逻辑处理输入数据，和训练时完全一致

---

### 四、TF Transform 的价值总结
- **解决了训练/服务不一致问题**：一套代码，两端使用，逻辑完全一致
- **提高了大数据预处理效率**：支持分布式批处理，训练前一次性处理完数据
- **降低了维护成本**：修改预处理逻辑时，只需要改一次代码，不用维护多套实现
- **无缝集成 TFX 生态**：可以和 Data API、TFRecords、Keras 预处理层配合，构建端到端的机器学习流水线

---

### 补充：TF Transform 适用场景
- 数据集非常大，单机处理不过来，需要分布式预处理
- 模型部署在移动/浏览器端，需要和训练时完全一致的预处理逻辑
- 预处理逻辑复杂，容易在多端实现时出错
- 属于 TFX 平台的一部分，在生产级机器学习项目中广泛使用

---

# 13.5 TensorFlow数据集项目

In [107]:
# 下载MNIST数据集
import tensorflow_datasets as tfds
dataset = tfds.load(name = 'mnist')
mnist_train,mnist_test = dataset['train'],dataset['test']

In [108]:
# 构建一个简单的模型
model = keras.Sequential([
    keras.layers.Flatten(input_shape=(28, 28)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(10)
])

# 定义优化器和损失函数
optimizer = keras.optimizers.Adam()
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

mnist_train = mnist_train.shuffle(10000).batch(32).prefetch(1)
# 遍历数据集训练
for item in mnist_train:
    images = item["image"]
    labels = item["label"]

    # 前向传播 + 计算损失 + 反向传播
    with tf.GradientTape() as tape:
        logits = model(images, training=True)
        loss_value = loss_fn(labels, logits)
    grads = tape.gradient(loss_value, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

In [109]:
# 使用map方法转换数据
mnist_train = mnist_train.shuffle(10000).batch(32)
mnist_train = mnist_train.map(lambda items:(items["image"],items["label"]))
model_train = mnist_train.prefetch(1)

In [110]:
dataset = tfds.load(name = 'mnist',batch_size = 32,as_supervised=True)
mnist_train = dataset['train'].prefetch(1)
model = keras.Sequential([
    keras.layers.Flatten(input_shape=(28, 28)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(10,activation='softmax')
])
model.compile(loss = 'sparse_categorical_crossentropy',optimizer = 'sgd')
model.fit(mnist_train,epochs = 5)

Epoch 1/5
1875/1875 [==============================] - 4s 2ms/step - loss: 19.6751
Epoch 2/5
1875/1875 [==============================] - 3s 2ms/step - loss: 1.8995
Epoch 3/5
1875/1875 [==============================] - 3s 2ms/step - loss: 2.0984
Epoch 4/5
1875/1875 [==============================] - 3s 2ms/step - loss: 2.0140
Epoch 5/5
1875/1875 [==============================] - 3s 2ms/step - loss: 1.8994
